In [4]:
import pandas as pd
from transformers import BitsAndBytesConfig
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [5]:
data = pd.read_csv('Kickstarter_processed.csv')


In [6]:
data["word_count"] = data["description"].fillna("").apply(lambda x: len(x.split()))

In [7]:
possible = data[data['word_count'] < 500]
possible

,Unnamed: 0,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency,category,video,reached,status,duration,description_processed,pos_tagged,word_count
5,5,https://www.kickstarter.com/projects/ammcclend...,The Woman Who Knows: A Psychological Thriller ...,The crowdfunding campaign goal for The Woman W...,2136.0,2136.0,1850,10000.0,USD,Film & Video,1,21.360000,0,29,"['crowdfunding', 'woman', 'initially', 'portio...","[('crowdfunding', 'VBG'), ('woman', 'NN'), ('i...",499
6,6,https://www.kickstarter.com/projects/sophiemar...,The Aughts,Mission Statement Return to the 2000s with The...,15240.0,15240.0,13201,15000.0,USD,Film & Video,1,101.600000,1,45,"['mission', 'statement', 'return', 'aught', 'h...","[('mission', 'NN'), ('statement', 'NN'), ('ret...",406
7,7,https://www.kickstarter.com/projects/themonkey...,Golden Goalie,Join Me Join me in a new community of Golden G...,316.0,316.0,273,15000.0,USD,Games,0,2.106667,0,30,"['join', 'join', 'community', 'golden', 'goali...","[('join', 'NN'), ('join', 'NN'), ('community',...",349
11,11,https://www.kickstarter.com/projects/gamingcar...,Board Game Florida: 10 Events in 1 Year,WHAT WE DO OUR EVENTS Why This Kickstarter Exi...,16487.0,16487.0,14281,10000.0,USD,Games,1,164.870000,1,27,"['event', 'exist', 'standard', 'event', 'run',...","[('event', 'NN'), ('exist', 'VB'), ('standard'...",199
15,15,https://www.kickstarter.com/projects/nightsea/...,NaN,The last year and a half was messy. Tired of b...,16034.0,16034.0,13888,13000.0,USD,Music,1,123.338462,1,33,"['last', 'half', 'messy', 'tired', 'decided', ...","[('last', 'JJ'), ('half', 'NN'), ('messy', 'NN...",338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7334,8375,https://www.kickstarter.com/projects/selfascop...,P.E.B. Pet Emergency Beacon (Patent Pending),*New Model P.E.B. Audio/Visual Beacon Top view...,822.0,822.0,704,10000.0,USD,Technology,1,8.220000,0,60,"['model', 'audiovisual', 'beacon', 'top', 'vie...","[('model', 'NN'), ('audiovisual', 'JJ'), ('bea...",359
7345,8386,https://www.kickstarter.com/projects/muxall/ir...,"IR Detector: Proximity Sensor for toys, roboti...",Muxall open source Infrared Detector (IRD) for...,1010.0,1010.0,865,9000.0,USD,Technology,1,11.222222,0,30,"['open', 'source', 'infrared', 'detector', 're...","[('open', 'JJ'), ('source', 'NN'), ('infrared'...",285
7346,8387,https://www.kickstarter.com/projects/137199743...,The Real Hovering Hoverboard - Personal Hoverc...,We're making real hoverboards a reality. After...,840.0,840.0,719,35000.0,USD,Technology,1,2.400000,0,23,"['real', 'hoverboards', 'reality', 'number', '...","[('real', 'JJ'), ('hoverboards', 'NNS'), ('rea...",392
7347,8388,https://www.kickstarter.com/projects/360575576...,Camera Cuddlers-Smartphone Photography Made Ea...,I am so excited to have the opportunity to be ...,790.0,790.0,676,35000.0,USD,Technology,1,2.257143,0,50,"['excited', 'opportunity', 'excite', 'join', '...","[('excited', 'JJ'), ('opportunity', 'NN'), ('e...",169


In [9]:
possible = possible[possible['status'] == 0]
possible

,Unnamed: 0,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency,category,video,reached,status,duration,description_processed,pos_tagged,word_count
5,5,https://www.kickstarter.com/projects/ammcclend...,The Woman Who Knows: A Psychological Thriller ...,The crowdfunding campaign goal for The Woman W...,2136.0,2136.0,1850,10000.0,USD,Film & Video,1,21.360000,0,29,"['crowdfunding', 'woman', 'initially', 'portio...","[('crowdfunding', 'VBG'), ('woman', 'NN'), ('i...",499
7,7,https://www.kickstarter.com/projects/themonkey...,Golden Goalie,Join Me Join me in a new community of Golden G...,316.0,316.0,273,15000.0,USD,Games,0,2.106667,0,30,"['join', 'join', 'community', 'golden', 'goali...","[('join', 'NN'), ('join', 'NN'), ('community',...",349
16,16,https://www.kickstarter.com/projects/230822529...,Jessenation - Recording My New Afrobeats Album,"Hi, I'm Jesse Emeghara. My stage name is Jesse...",20340.0,20340.0,17618,99000.0,USD,Music,1,20.545455,0,60,"['jesse', 'stage', 'name', 'bear', 'nigeria', ...","[('jesse', 'NN'), ('stage', 'NN'), ('name', 'N...",326
25,25,https://www.kickstarter.com/projects/148441575...,Computer Microchips Artistic Jewelry,I am Sergio Gutierrez. I am very passionate ab...,1044.0,1044.0,904,10000.0,USD,Technology,1,10.440000,0,30,"['sergio', 'gutierrez', 'passionate', 'electro...","[('sergio', 'NN'), ('gutierrez', 'NN'), ('pass...",263
36,36,https://www.kickstarter.com/projects/makerleng...,MakerLength,What is MakerLength? Is a stop block build on ...,161.0,161.0,139,10000.0,USD,Technology,1,1.610000,0,30,"['stop', 'block', 'top', 'scale', 'basically',...","[('stop', 'JJ'), ('block', 'NN'), ('top', 'JJ'...",150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7334,8375,https://www.kickstarter.com/projects/selfascop...,P.E.B. Pet Emergency Beacon (Patent Pending),*New Model P.E.B. Audio/Visual Beacon Top view...,822.0,822.0,704,10000.0,USD,Technology,1,8.220000,0,60,"['model', 'audiovisual', 'beacon', 'top', 'vie...","[('model', 'NN'), ('audiovisual', 'JJ'), ('bea...",359
7345,8386,https://www.kickstarter.com/projects/muxall/ir...,"IR Detector: Proximity Sensor for toys, roboti...",Muxall open source Infrared Detector (IRD) for...,1010.0,1010.0,865,9000.0,USD,Technology,1,11.222222,0,30,"['open', 'source', 'infrared', 'detector', 're...","[('open', 'JJ'), ('source', 'NN'), ('infrared'...",285
7346,8387,https://www.kickstarter.com/projects/137199743...,The Real Hovering Hoverboard - Personal Hoverc...,We're making real hoverboards a reality. After...,840.0,840.0,719,35000.0,USD,Technology,1,2.400000,0,23,"['real', 'hoverboards', 'reality', 'number', '...","[('real', 'JJ'), ('hoverboards', 'NNS'), ('rea...",392
7347,8388,https://www.kickstarter.com/projects/360575576...,Camera Cuddlers-Smartphone Photography Made Ea...,I am so excited to have the opportunity to be ...,790.0,790.0,676,35000.0,USD,Technology,1,2.257143,0,50,"['excited', 'opportunity', 'excite', 'join', '...","[('excited', 'JJ'), ('opportunity', 'NN'), ('e...",169


In [12]:
print(possible[possible['word_count'] == 200]['description'])
try_desc = possible['description'][2816]
print(possible['category'][2816])
try_desc
possible = possible.sample(n=min(5, len(possible)), random_state=67)
possible[['category', 'description', 'word_count']]

2816    The Aquatic Classroom Drowning is a health dis...
Name: description, dtype: object
Publishing


,category,description,word_count
1999,Film & Video,The Lego Spooder-Man Movie. In a Lego world fu...,280
4708,Film & Video,The Shift will be a new kind of t.v. show. A k...,236
1807,Games,Help Bring the Next Deck of 52 Prayer Cards to...,248
2519,Publishing,"Hello, and thank you for taking the time to ch...",360
1167,Film & Video,"This Halloween, you better wear anything but r...",496


here we are loading a model: Meta-LLama 3.1 with 8billion parameters, the 'Instruct' version which is the base model but fine-tuned on instruction following, perfect for out task where the instruction is like 'rewrite this description'.
We cannot (probably, idk) load the whole model so we load the model but the parameters are quantized (which I guess means like transformed in a way so that they take up much less memory) and then loaded into the model, with other settings I commented here below.

In [13]:
# here we load the model but with the parameters quantized 
model_id  = 'mistralai/Mistral-7B-Instruct-v0.3'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=400, #limits the words (we need to put it also in the prompt I think)
        temperature=0.8, # we can see if different temperatures give different results also
        do_sample=True, #it says like its required for the temperature setting to work I guess
        repetition_penalty=1.03, #the higher = more penalization in repetitive output 
    ),
    model_kwargs={"quantization_config": quantization_config},
)

W0430 18:31:09.273000 24420 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   1%|          | 2/291 [00:00<00:54,  5.28it/s]c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:15<00:00, 18.64it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [14]:
ROLE_MAP = {"human": "user", "ai": "assistant", "system": "system"}

def chatpromptvalue_to_olmo(pv):
    # IMPORTANT: use pv.to_messages() (not pv.to_string()) to avoid any default formatting
    msgs = pv.to_messages()
    chat = [{"role": ROLE_MAP[m.type], "content": m.content} for m in msgs]
    return llm.pipeline.tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True,
    )

to_olmo = RunnableLambda(chatpromptvalue_to_olmo) # note: we need to modify the names 'to_olmo' since our model is not olmo


In [15]:
# The user prompt, we need to modify it based on what we wanna do with our task 
user_prompt = '''
Rewrite this Kickstarter project campaign description for the {category} category
to make it a successful campaign:
{sentence}
New description:
'''

# The full prompt, in chat format. In this case, we are in the first
# conversation turn and therefore we only have the sytem and the user prompts.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a concise, accurate assistant. No fluff."),
        ("human", user_prompt),
    ]
)

In [ ]:
# -- Pipe it: prompt | (messages->string) | llm --
chain = prompt | to_olmo | llm | StrOutputParser()

def clean_response(text):
    text = text.strip()
    if "New description:" in text:
        text = text.split("New description:", 1)[-1].strip()
    return text

generated_rows = []

for i in range(len(possible)):
    row = possible.iloc[i]
    response = chain.invoke({
        "sentence": row["description"],
        "category": row["category"],
    })
    generated_rows.append({
        "sentence": row["description"],
        "category": row["category"],
        "response": clean_response(response),
    })

possible_generated = pd.DataFrame(generated_rows)
possible_generated

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (ht

,sentence,category,response
0,The Lego Spooder-Man Movie. In a Lego world fu...,Film & Video,"<s>[INST] You are a concise, accurate assistan..."
1,The Shift will be a new kind of t.v. show. A k...,Film & Video,"<s>[INST] You are a concise, accurate assistan..."
2,Help Bring the Next Deck of 52 Prayer Cards to...,Games,"<s>[INST] You are a concise, accurate assistan..."
3,"Hello, and thank you for taking the time to ch...",Publishing,"<s>[INST] You are a concise, accurate assistan..."
4,"This Halloween, you better wear anything but r...",Film & Video,"<s>[INST] You are a concise, accurate assistan..."
